In [1]:
import pandas as pd
import numpy as np
from itertools import combinations
from scipy.stats import norm

In [10]:
def get_put_spreads(min_pop=0.5, max_net_delta=999, min_net_theta=25.0, max_net_vega=0.0, 
                    max_net_gamma=0.0, min_credit=2.0, max_width=200, min_rr=0.01, 
                    min_max_profit=100.0, max_max_profit=1000000, min_max_loss=200.0, max_max_loss=100000.0, lot_size=65):
    all_results = []
    df = pd.read_csv("../dashboard/data/28_Apr_2026_PE.csv")
    df = df.sort_values('strike', ascending=False).reset_index(drop=True)
    for (i, row_a), (j, row_b) in combinations(df.iterrows(), 2):
        # Assign short (higher strike) and long (lower strike)
        if row_a['strike'] > row_b['strike']:
            short, long = row_a, row_b
        else:
            short, long = row_b, row_a

        strike_width = short['strike'] - long['strike']
        if strike_width > max_width:
            continue
        if strike_width <= 0:
            continue

        short_mid = (short['bid'] + short['ask']) / 2
        long_mid  = (long['bid']  + long['ask'])  / 2
        net_credit  = (short_mid - long_mid) * lot_size
        max_loss    = (strike_width - (short_mid - long_mid)) * lot_size
        reward_risk = net_credit / max_loss if max_loss > 0 else 0

        # Put deltas are negative; negate short to get position delta
        # net_delta ends up positive (spread profits when market rises / stays flat)
        net_delta = (-short['delta'] + long['delta']) * lot_size
        net_theta = (-short['theta'] + long['theta']) * lot_size
        net_vega  = (-short['vega']  + long['vega'])  * lot_size
        net_gamma = (-short['gamma'] + long['gamma']) * lot_size

        # POP = probability short put expires worthless = 1 - |short_delta|
        pop = 1 - abs(short['delta'])

        # ── Filters ──────────────────────────────────────────────────────
        if max_loss <= 0:
            continue
        if net_credit <= 0:
            continue
        if not (min_max_profit <= net_credit <= max_max_profit):
            continue
        if not (min_max_loss <= max_loss <= max_max_loss):
            continue
        if net_credit < min_credit:
            continue
        if reward_risk < min_rr:
            continue
        if abs(net_delta) > max_net_delta:
            continue
        if net_theta < min_net_theta:
            continue
        if net_vega > max_net_vega:
            continue
        if net_gamma > max_net_gamma:
            continue
        if pop < min_pop:
            continue
        # Short-leg delta must be in the target range [-0.30, -0.08]
        if not (-0.30 <= short['delta'] <= -0.08):
            continue

        all_results.append({
            'expiry':       "28_Apr_2026",
            'short_strike': short['strike'],   # higher strike (sold)
            'long_strike':  long['strike'],    # lower  strike (bought)
            'short_token':  short['token'],
            'long_token':   long['token'],
            'width':        round(strike_width, 2),
            'net_credit':   round(net_credit, 2),
            'max_profit':   round(net_credit, 2),
            'max_loss':     round(max_loss, 2),
            'reward_risk':  round(reward_risk, 4),
            'net_delta':    round(net_delta, 4),
            'net_theta':    round(net_theta, 4),
            'net_vega':     round(net_vega, 4),
            'net_gamma':    round(net_gamma, 6),
            'pop':          round(pop, 4),
        })
    if not all_results:
        return pd.DataFrame()
    return pd.DataFrame(all_results).sort_values('reward_risk', ascending=False).reset_index(drop=True)



In [11]:
puts = get_put_spreads()
puts

,expiry,short_strike,long_strike,short_token,long_token,width,net_credit,max_profit,max_loss,reward_risk,net_delta,net_theta,net_vega,net_gamma,pop
0,28_Apr_2026,23350.0,23200.0,72223,72217,150.0,2250.62,2250.62,7499.38,0.3001,2.2100,29.8545,-73.1185,0.0,0.7029
1,28_Apr_2026,23350.0,23150.0,72223,72215,200.0,2934.75,2934.75,10065.25,0.2916,3.1525,44.5575,-107.3085,0.0,0.7029
2,28_Apr_2026,23300.0,23150.0,72221,72215,150.0,2112.50,2112.50,7637.50,0.2766,2.5285,36.6795,-87.4575,0.0,0.7125
3,28_Apr_2026,23300.0,23100.0,72221,72213,200.0,2684.50,2684.50,10315.50,0.2602,3.1330,47.3720,-111.6245,0.0,0.7125
4,28_Apr_2026,23250.0,23100.0,72219,72213,150.0,1912.63,1912.63,7837.37,0.2440,2.0995,33.9365,-78.5330,0.0,0.7284
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,28_Apr_2026,22250.0,22050.0,71960,71952,200.0,885.62,885.62,12114.38,0.0731,1.3390,57.4210,-107.7505,0.0,0.9124
64,28_Apr_2026,22250.0,22100.0,71960,71954,150.0,646.75,646.75,9103.25,0.0710,1.0075,42.5035,-79.9240,0.0,0.9124
65,28_Apr_2026,22200.0,22050.0,71958,71952,150.0,643.50,643.50,9106.50,0.0707,1.0595,45.7665,-85.7545,0.0,0.9167
66,28_Apr_2026,22200.0,22100.0,71958,71954,100.0,404.63,404.63,6095.38,0.0664,0.7280,30.8490,-57.9280,0.0,0.9167
